In [111]:
pip install finrl yfinance stable-baselines3 gym pandas numpy matplotlib seaborn

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.comNote: you may need to restart the kernel to use updated packages.



In [112]:
import numpy as np
import pandas as pd
import yfinance as yf
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from stable_baselines3 import PPO, A2C, DDPG, SAC, TD3
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback



In [117]:
import yfinance as yf
import pandas as pd

# Define date range
start_date = "2010-01-01"
end_date = "2024-12-31"

# List of stocks in the Dow Jones 30 (excluding 'DOW' and 'UTX')
tickers = [
    'MMM', 'AXP', 'AAPL', 'BA', 'CAT', 'CVX', 'CSCO', 'KO', 'DIS', 'GS',
    'HD', 'IBM', 'INTC', 'JNJ', 'JPM', 'MCD', 'MRK', 'MSFT', 'NKE',
    'PFE', 'PG', 'TRV', 'UNH', 'VZ', 'V', 'WBA', 'WMT', 'XOM'
]

# Define the required column format
expected_columns = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume", "Ticker"]

# Function to fetch stock data with error handling
def fetch_stock_data(tickers, start_date, end_date):
    stock_data = []
    failed_tickers = []  # Track tickers that failed

    for ticker in tickers:
        try:
            data = yf.download(ticker, start=start_date, end=end_date)
            if data.empty:
                print(f"⚠️ Warning: No data found for {ticker}, skipping...")
                failed_tickers.append(ticker)
            else:
                data["Ticker"] = ticker  # Add ticker column
                data.reset_index(inplace=True)  # Reset index to include Date
                stock_data.append(data)
                print(f"✅ Successfully fetched data for {ticker}")
        except Exception as e:
            print(f"❌ Error fetching {ticker}: {e}")
            failed_tickers.append(ticker)

    print("\n⏳ Retrying failed tickers...")
    for ticker in failed_tickers:
        try:
            data = yf.download(ticker, start=start_date, end=end_date)
            if not data.empty:
                data["Ticker"] = ticker
                data.reset_index(inplace=True)
                stock_data.append(data)
                print(f"✅ Successfully fetched {ticker} on retry")
            else:
                print(f"🚨 Skipping {ticker}, data still unavailable.")
        except Exception as e:
            print(f"❌ Final attempt failed for {ticker}: {e}")

    return stock_data

# Fetch stock data with error handling
df_list = fetch_stock_data(tickers, start_date, end_date)

# Combine all stock data into one DataFrame
if df_list:
    df = pd.concat(df_list, ignore_index=True)
    print("\n✅ Combined dataset shape:", df.shape)

    # Ensure column order is consistent and select only expected columns
    df = df[[col for col in expected_columns if col in df.columns]]

    # Drop missing values (if any)
    df.dropna(inplace=True)
    
    # Display final dataset
    print("\nFinal dataset preview:\n", df.head())
else:
    print("\n🚨 No data available! Check API or ticker list.")


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['AAPL']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
[*********************100%***********************]  1 of 1 completed


✅ Successfully fetched data for MMM
✅ Successfully fetched data for AXP
⚠️ Warning: No data found for AAPL, skipping...
✅ Successfully fetched data for BA


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


✅ Successfully fetched data for CAT
✅ Successfully fetched data for CVX
✅ Successfully fetched data for CSCO


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


...

✅ Successfully fetched data for PG
✅ Successfully fetched data for TRV
✅ Successfully fetched data for UNH


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['V']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
[*********************100%***********************]  1 of 1 completed


✅ Successfully fetched data for VZ
⚠️ Warning: No data found for V, skipping...
✅ Successfully fetched data for WBA


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['AAPL']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['IBM']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['V']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')


✅ Successfully fetched data for WMT
✅ Successfully fetched data for XOM

⏳ Retrying failed tickers...
🚨 Skipping AAPL, data still unavailable.
🚨 Skipping IBM, data still unavailable.
🚨 Skipping V, data still unavailable.

✅ Combined dataset shape: (94325, 127)

Final dataset preview:
 Empty DataFrame
Columns: [(Date, ), (Open, MMM), (Open, AXP), (Open, BA), (Open, CAT), (Open, CVX), (Open, CSCO), (Open, KO), (Open, DIS), (Open, GS), (Open, HD), (Open, INTC), (Open, JNJ), (Open, JPM), (Open, MCD), (Open, MRK), (Open, MSFT), (Open, NKE), (Open, PFE), (Open, PG), (Open, TRV), (Open, UNH), (Open, VZ), (Open, WBA), (Open, WMT), (Open, XOM), (High, MMM), (High, AXP), (High, BA), (High, CAT), (High, CVX), (High, CSCO), (High, KO), (High, DIS), (High, GS), (High, HD), (High, INTC), (High, JNJ), (High, JPM), (High, MCD), (High, MRK), (High, MSFT), (High, NKE), (High, PFE), (High, PG), (High, TRV), (High, UNH), (High, VZ), (High, WBA), (High, WMT), (High, XOM), (Low, MMM), (Low, AXP), (Low, BA),

In [93]:
'''
# List of stocks in the Dow Jones 30
tickers = [
    'MMM', 'AXP', 'AAPL', 'BA', 'CAT', 'CVX', 'CSCO', 'KO', 'DIS', 'DOW',
    'GS', 'HD', 'IBM', 'INTC', 'JNJ', 'JPM', 'MCD', 'MRK', 'MSFT', 'NKE',
    'PFE', 'PG', 'TRV', 'UNH', 'UTX', 'VZ', 'V', 'WBA', 'WMT', 'XOM'
]
tickers.remove('DOW')

tickers.remove('UTX')

# Get historical data from Yahoo Finance and save it to dictionary
def fetch_stock_data(tickers, start_date, end_date):
    stock_data = {}
    for ticker in tickers:
        stock_data[ticker] = yf.download(ticker, start=start_date, end=end_date)
    return stock_data

# Call the function to get data
stock_data = fetch_stock_data(tickers, '2010-01-01', '2024-12-31')
'''

"\n# List of stocks in the Dow Jones 30\ntickers = [\n    'MMM', 'AXP', 'AAPL', 'BA', 'CAT', 'CVX', 'CSCO', 'KO', 'DIS', 'DOW',\n    'GS', 'HD', 'IBM', 'INTC', 'JNJ', 'JPM', 'MCD', 'MRK', 'MSFT', 'NKE',\n    'PFE', 'PG', 'TRV', 'UNH', 'UTX', 'VZ', 'V', 'WBA', 'WMT', 'XOM'\n]\ntickers.remove('DOW')\n\ntickers.remove('UTX')\n\n# Get historical data from Yahoo Finance and save it to dictionary\ndef fetch_stock_data(tickers, start_date, end_date):\n    stock_data = {}\n    for ticker in tickers:\n        stock_data[ticker] = yf.download(ticker, start=start_date, end=end_date)\n    return stock_data\n\n# Call the function to get data\nstock_data = fetch_stock_data(tickers, '2010-01-01', '2024-12-31')\n"

In [91]:
'''import yfinance as yf
import pandas as pd

# Define date range
start_date = "2010-01-01"
end_date = "2024-12-31"

# List of stock tickers (excluding 'DOW' and 'UTX')
ticker_list = [
    'MMM', 'AXP', 'AAPL', 'BA', 'CAT', 'CVX', 'CSCO', 'KO', 'DIS', 'GS',
    'HD', 'IBM', 'INTC', 'JNJ', 'JPM', 'MCD', 'MRK', 'MSFT', 'NKE',
    'PFE', 'PG', 'TRV', 'UNH', 'VZ', 'V', 'WBA', 'WMT', 'XOM'
]

# Define the required column format
expected_columns = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume", "Ticker"]

# Initialize an empty list to store individual DataFrames for each stock
df_list = []

# Loop through each ticker and fetch data separately
for ticker in ticker_list:
    try:
        stock_data = yf.download(ticker, start=start_date, end=end_date)
        stock_data["Ticker"] = ticker  # Add a column for the ticker symbol
        stock_data.reset_index(inplace=True)  # Reset the index so 'Date' is a column
        df_list.append(stock_data)
        print(f"Successfully fetched data for {ticker}")
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")

# Combine all stock data into one DataFrame
df = pd.concat(df_list, ignore_index=True)

# Print the actual column names in the DataFrame for debugging
print("Actual DataFrame columns:", df.columns)

# Ensure column order is consistent and select only the expected columns
df = df[[col for col in expected_columns if col in df.columns]]  # Select columns present in the DataFrame

# Drop missing values (if any)
df.dropna(inplace=True)

# Display final dataset
print(df.head())
'''

'import yfinance as yf\nimport pandas as pd\n\n# Define date range\nstart_date = "2010-01-01"\nend_date = "2024-12-31"\n\n# List of stock tickers (excluding \'DOW\' and \'UTX\')\nticker_list = [\n    \'MMM\', \'AXP\', \'AAPL\', \'BA\', \'CAT\', \'CVX\', \'CSCO\', \'KO\', \'DIS\', \'GS\',\n    \'HD\', \'IBM\', \'INTC\', \'JNJ\', \'JPM\', \'MCD\', \'MRK\', \'MSFT\', \'NKE\',\n    \'PFE\', \'PG\', \'TRV\', \'UNH\', \'VZ\', \'V\', \'WBA\', \'WMT\', \'XOM\'\n]\n\n# Define the required column format\nexpected_columns = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume", "Ticker"]\n\n# Initialize an empty list to store individual DataFrames for each stock\ndf_list = []\n\n# Loop through each ticker and fetch data separately\nfor ticker in ticker_list:\n    try:\n        stock_data = yf.download(ticker, start=start_date, end=end_date)\n        stock_data["Ticker"] = ticker  # Add a column for the ticker symbol\n        stock_data.reset_index(inplace=True)  # Reset the index so \'Da

## Download and combine data

In [18]:
import yfinance as yf
import pandas as pd

# Define date range
start_date = "2010-01-01"
end_date = "2024-12-31"

# List of stock tickers
ticker_list = [
    'MMM', 'AXP', 'AAPL', 'BA', 'CAT', 'CVX', 'CSCO', 'KO', 'DIS', 'GS',
    'HD', 'IBM', 'INTC', 'JNJ', 'JPM', 'MCD', 'MRK', 'MSFT', 'NKE',
    'PFE', 'PG', 'TRV', 'UNH', 'VZ', 'V', 'WBA', 'WMT', 'XOM'
]

# Fetch all tickers at once
stock_data = yf.download(ticker_list, start=start_date, end=end_date, group_by='ticker')

# Flatten MultiIndex and reformat DataFrame
flattened_data = []

for ticker in ticker_list:
    try:
        temp_df = stock_data[ticker].copy()
        temp_df["Ticker"] = ticker  # Add a column for the ticker symbol
        temp_df.reset_index(inplace=True)  # Convert index to a column (Date)
        flattened_data.append(temp_df)
        print(f"✅ Successfully fetched data for {ticker}")
    except Exception as e:
        print(f"⚠️ Error fetching data for {ticker}: {e}")

# Combine all individual DataFrames into one
df = pd.concat(flattened_data, ignore_index=True)

# Rename columns for consistency
df.rename(columns={"Adj Close": "Adj_Close"}, inplace=True)

# Drop missing values (if any)
df.dropna(inplace=True)

# Ensure proper column order
expected_columns = ["Date", "Open", "High", "Low", "Close", "Adj_Close", "Volume", "Ticker"]
df = df[[col for col in expected_columns if col in df.columns]]

# Display first few rows
print("Final DataFrame:")
print(df.head())
print(df.tail())
# Save to CSV for future use
df.to_csv("dow_jones_stock_data.csv", index=False)


[*********************100%***********************]  28 of 28 completed

3 Failed downloads:
['V', 'IBM', 'AAPL']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')


✅ Successfully fetched data for MMM
✅ Successfully fetched data for AXP
✅ Successfully fetched data for AAPL
✅ Successfully fetched data for BA
✅ Successfully fetched data for CAT
✅ Successfully fetched data for CVX
✅ Successfully fetched data for CSCO
✅ Successfully fetched data for KO
✅ Successfully fetched data for DIS
✅ Successfully fetched data for GS
✅ Successfully fetched data for HD
✅ Successfully fetched data for IBM
✅ Successfully fetched data for INTC
✅ Successfully fetched data for JNJ
✅ Successfully fetched data for JPM
✅ Successfully fetched data for MCD
✅ Successfully fetched data for MRK
✅ Successfully fetched data for MSFT
✅ Successfully fetched data for NKE
✅ Successfully fetched data for PFE
✅ Successfully fetched data for PG
✅ Successfully fetched data for TRV
✅ Successfully fetched data for UNH
✅ Successfully fetched data for VZ
✅ Successfully fetched data for V
✅ Successfully fetched data for WBA
✅ Successfully fetched data for WMT
✅ Successfully fetched data for 

## PREPROCESSING

In [21]:
import pandas as pd

# Load the stock data (assuming `df` is already fetched)
df.dropna(inplace=True)  # Drop missing values (alternative: df.fillna(method='ffill'))

# Convert Date to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Sort data by Ticker and Date
df = df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

### ------------------- Feature Engineering -------------------

# Function to calculate Moving Averages
def moving_average(df, window):
    df[f'MA_{window}'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=window).mean())

# Function to calculate Relative Strength Index (RSI)
def compute_rsi(df, window=14):
    delta = df.groupby('Ticker')['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(window=window).mean()
    loss = -delta.where(delta < 0, 0).rolling(window=window).mean()
    rs = gain / loss
    df[f'RSI_{window}'] = 100 - (100 / (1 + rs))

# Function to calculate MACD (Moving Average Convergence Divergence)
def compute_macd(df):
    short_ema = df.groupby('Ticker')['Close'].transform(lambda x: x.ewm(span=12, adjust=False).mean())
    long_ema = df.groupby('Ticker')['Close'].transform(lambda x: x.ewm(span=26, adjust=False).mean())
    df['MACD'] = short_ema - long_ema
    df['MACD_Signal'] = df.groupby('Ticker')['MACD'].transform(lambda x: x.ewm(span=9, adjust=False).mean())

# Function to calculate Bollinger Bands
def compute_bollinger_bands(df, window=20):
    rolling_mean = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=window).mean())
    rolling_std = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=window).std())
    df['BB_Upper'] = rolling_mean + (rolling_std * 2)
    df['BB_Lower'] = rolling_mean - (rolling_std * 2)

# Apply all feature calculations
moving_average(df, window=10)
moving_average(df, window=50)
compute_rsi(df, window=14)
compute_macd(df)
compute_bollinger_bands(df, window=20)

# Normalize price values (Min-Max Scaling) for ML models
df['Close_Norm'] = df.groupby('Ticker')['Close'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))

# Drop missing values after calculations (important for avoiding NaNs from rolling calculations)
df.dropna(inplace=True)

# Save preprocessed data to CSV
df.to_csv('preprocessed_stock_data.csv', index=False)

# Display first few rows
print("Preprocessed Stock Data Sample:")
print(df.head())
print(df.tail())


Preprocessed Stock Data Sample:
Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj_Close, Volume, Ticker, MA_10, MA_50, RSI_14, MACD, MACD_Signal, BB_Upper, BB_Lower, Close_Norm]
Index: []
Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj_Close, Volume, Ticker, MA_10, MA_50, RSI_14, MACD, MACD_Signal, BB_Upper, BB_Lower, Close_Norm]
Index: []


In [23]:
# Remove initial rows with NaN values from moving averages, RSI, and Bollinger Bands
df = df.dropna().reset_index(drop=True)

In [25]:
pip install alpaca_trade_api

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [27]:
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
import pandas as pd

## implementing Risk Analysis with DDGP

In [29]:
import pandas as pd
import yfinance as yf

# Define Date Range (Ensure alignment with stock data)
start_date = "2010-01-01"
end_date = "2024-12-31"

# Fetch VIX Data
vix_data = yf.download("^VIX", start=start_date, end=end_date)

# Flatten MultiIndex if present
if isinstance(vix_data.columns, pd.MultiIndex):
    vix_data.columns = [col[0] for col in vix_data.columns]  # Keep only first level

# Reset index and rename 'Date' to match stock data format
vix_data = vix_data.reset_index()
vix_data.rename(columns={"Date": "date"}, inplace=True)
vix_data["date"] = pd.to_datetime(vix_data["date"])

# Ensure stock data's 'Date' column is in datetime format
df["Date"] = pd.to_datetime(df["Date"])

# Debugging: Print column names before merging
print("df columns:", df.columns)
print("vix_data columns:", vix_data.columns)

# Merge VIX data with stock data using 'Date' from df and 'date' from vix_data
df = pd.merge(df, vix_data, left_on="Date", right_on="date", how="left")

# Drop extra 'date' column after merging
df.drop(columns=["date"], inplace=True)

print("✅ Merge successful!")


[*********************100%***********************]  1 of 1 completed

df columns: Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj_Close', 'Volume', 'Ticker',
       'MA_10', 'MA_50', 'RSI_14', 'MACD', 'MACD_Signal', 'BB_Upper',
       'BB_Lower', 'Close_Norm'],
      dtype='object', name='Price')
vix_data columns: Index(['date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')
✅ Merge successful!


In [236]:
%whos

Variable                                 Type           Data/Info
-----------------------------------------------------------------
A2C                                      ABCMeta        <class 'stable_baselines3.a2c.a2c.A2C'>
A2CAgent                                 type           <class '__main__.A2CAgent'>
BaseCallback                             ABCMeta        <class 'stable_baselines3<...>.callbacks.BaseCallback'>
DDPG                                     ABCMeta        <class 'stable_baselines3.ddpg.ddpg.DDPG'>
DDPGAgent                                type           <class '__main__.DDPGAgent'>
DummyVecEnv                              ABCMeta        <class 'stable_baselines3<...>mmy_vec_env.DummyVecEnv'>
EnsembleAgent                            type           <class '__main__.EnsembleAgent'>
PPO                                      ABCMeta        <class 'stable_baselines3.ppo.ppo.PPO'>
PPOAgent                                 type           <class '__main__.PPOAgent'>
PolicyGradi

In [32]:
import pandas as pd
import yfinance as yf

# Fetch VIX Data
start_date = "2010-01-01"
end_date = "2024-12-31"
vix_data = yf.download("^VIX", start=start_date, end=end_date)

# Ensure df_vix is a single-index DataFrame
if isinstance(vix_data.columns, pd.MultiIndex):
    vix_data.columns = vix_data.columns.get_level_values(0)  # Flatten MultiIndex

# Reset index and rename columns
df_vix = vix_data.reset_index()
df_vix.rename(columns={"Date": "date", "Close": "vix"}, inplace=True)

# Drop duplicates and NaNs
df_vix.drop_duplicates(inplace=True)
df_vix.dropna(inplace=True)

# Ensure date format consistency
df_vix["date"] = pd.to_datetime(df_vix["date"])
df["Date"] = pd.to_datetime(df["Date"])

# Debugging: Print column levels to verify
print("df_vix column levels:", df_vix.columns.nlevels)  # Should be 1
print("df column levels:", df.columns.nlevels)  # Should be 1

# Merge VIX data with stock data
df = df.merge(df_vix, left_on="Date", right_on="date", how="left")

# Drop extra 'date' column after merging
df.drop(columns=["date"], inplace=True)

# Forward fill missing VIX values
df["vix"] = df["vix"].ffill()

print("✅ Successfully merged VIX data!")
print(df.head())  # Display first few rows


[*********************100%***********************]  1 of 1 completed

df_vix column levels: 1
df column levels: 1
✅ Successfully merged VIX data!
Empty DataFrame
Columns: [Date, Open_x, High_x, Low_x, Close_x, Adj_Close, Volume_x, Ticker, MA_10, MA_50, RSI_14, MACD, MACD_Signal, BB_Upper, BB_Lower, Close_Norm, Close_y, High_y, Low_y, Open_y, Volume_y, vix, High, Low, Open, Volume]
Index: []

[0 rows x 26 columns]


In [34]:
pip install exchange_calendars

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [35]:
pip install stockstats==0.4.0

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [37]:
pip install wrds

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [38]:

import numpy as np
import pandas as pd
import gym
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3 import PPO
from stable_baselines3.common.logger import configure
import torch


In [39]:
import numpy as np

def risk_adjusted_reward(trade_result, cash, portfolio_value, current_step, df):
    """
    Custom risk-adjusted reward function.
    - Applies penalties for high volatility, drawdowns, and risk factors.
    """

    # Base reward from trade result
    reward = trade_result

    # Apply Stop-Loss Penalty (if losses exceed threshold, punish heavily)
    if trade_result < -0.02 * portfolio_value:  # 2% loss
        reward -= abs(trade_result) * 1.5  # Increase penalty

    # Apply Take-Profit Bonus (if gains exceed threshold, reward)
    if trade_result > 0.02 * portfolio_value:  # 2% gain
        reward += abs(trade_result) * 0.5  # Moderate reward

    # Ensure VIX is available (avoid index errors)
    if "vix" in df.columns and current_step < len(df):
        vix_value = df["vix"].iloc[current_step] / 100  # Normalize VIX
        reward -= vix_value  # Higher VIX = Higher risk = Reduce reward
    else:
        vix_value = 0  # Default if VIX is missing

    # Sharpe Ratio Adjustment (reward stability)
    if current_step > 20:  # Ensure enough data points
        returns = df["Close"].pct_change().dropna()[:current_step]
        if len(returns) > 1:
            sharpe_ratio = np.mean(returns) / (np.std(returns) + 1e-6)  # Avoid div by zero
            reward += sharpe_ratio * 0.1  # Small bonus for stable returns

    return reward


In [41]:
class StockTradingRiskEnv(StockTradingEnv):
    def __init__(self, df, **kwargs):
        super().__init__(df, **kwargs)

    def _calculate_reward(self, actions):
        """
        Override reward calculation with risk-adjusted function.
        """
        # Ensure there are at least 2 values in portfolio history
        if len(self.account_information["portfolio_value"]) < 2:
            trade_result = 0  # No trade yet, no reward
        else:
            trade_result = self.account_information["portfolio_value"][-1] - self.account_information["portfolio_value"][-2]

        portfolio_value = self.account_information["portfolio_value"][-1]
        
        # Ensure cash history exists before accessing last value
        cash = self.account_information["cash"][-1] if "cash" in self.account_information else 0
        
        # Pass the current step to fetch VIX correctly
        return risk_adjusted_reward(trade_result, cash, portfolio_value, self.current_step, self.df)


## StockTradingRisk Environment

In [47]:
class StockTradingRiskEnv(StockTradingEnv):
    def __init__(self, df, stock_dim, hmax, initial_amount, num_stock_shares,
                 buy_cost_pct, sell_cost_pct, reward_scaling, state_space,
                 action_space, tech_indicator_list):
        super().__init__(df=df, stock_dim=stock_dim, hmax=hmax,
                         initial_amount=initial_amount, num_stock_shares=num_stock_shares,
                         buy_cost_pct=buy_cost_pct, sell_cost_pct=sell_cost_pct,
                         reward_scaling=reward_scaling, state_space=state_space,
                         action_space=action_space, tech_indicator_list=tech_indicator_list)

    def _calculate_reward(self, actions):
        """
        Override reward calculation with risk-adjusted function.
        """

        # ✅ Ensure at least 2 portfolio values exist before computing trade result
        if len(self.account_information["portfolio_value"]) < 2:
            trade_result = 0  # No previous value, no reward calculation
        else:
            trade_result = self.account_information["portfolio_value"][-1] - self.account_information["portfolio_value"][-2]

        portfolio_value = self.account_information["portfolio_value"][-1]

        # ✅ Ensure "cash" exists, otherwise use initial amount
        cash = self.account_information["cash"][-1] if "cash" in self.account_information else self.initial_amount

        # ✅ Pass `self.current_step` and `self.df` for VIX-based risk management
        return risk_adjusted_reward(trade_result, cash, portfolio_value, self.current_step, self.df)


In [49]:
# Check if the column exists and if the column is of the correct type
if "Close" in df.columns:
    if df["Close"].dtype != np.float64:
        df["Close"] = df["Close"].astype(np.float64)  # Ensure 'Close' is of float64 type
elif "close" in df.columns:
    if df["close"].dtype != np.float64:
        df["close"] = df["close"].astype(np.float64)  # Ensure 'close' is of float64 type
else:
    print("Error: 'Close' or 'close' column not found in DataFrame")

# In case you want to check and process if the 'Close' or 'close' column is a single scalar value
if "Close" in df.columns and isinstance(df["Close"].iloc[0], np.float64):
    df["Close"] = pd.Series([df["Close"]])

# Alternative for lowercase 'close' column
if "close" in df.columns and isinstance(df["close"].iloc[0], np.float64):
    df["close"] = pd.Series([df["close"]])

print("Processed DataFrame:")
print(df.head())


Error: 'Close' or 'close' column not found in DataFrame
Processed DataFrame:
Empty DataFrame
Columns: [Date, Open_x, High_x, Low_x, Close_x, Adj_Close, Volume_x, Ticker, MA_10, MA_50, RSI_14, MACD, MACD_Signal, BB_Upper, BB_Lower, Close_Norm, Close_y, High_y, Low_y, Open_y, Volume_y, vix, High, Low, Open, Volume]
Index: []

[0 rows x 26 columns]


In [51]:
# Reset the index of the DataFrame to ensure a clean DataFrame
df = df.reset_index(drop=True)  # Removes old index and creates a clean one

# Display the first few rows to confirm
print("Clean DataFrame:")
print(df.head())


Clean DataFrame:
Empty DataFrame
Columns: [Date, Open_x, High_x, Low_x, Close_x, Adj_Close, Volume_x, Ticker, MA_10, MA_50, RSI_14, MACD, MACD_Signal, BB_Upper, BB_Lower, Close_Norm, Close_y, High_y, Low_y, Open_y, Volume_y, vix, High, Low, Open, Volume]
Index: []

[0 rows x 26 columns]


In [53]:

print("Columns in df:", df.columns)  # Check column names
print("First few rows:\n", df.head())  # Check if 'close' is present

Columns in df: Index(['Date', 'Open_x', 'High_x', 'Low_x', 'Close_x', 'Adj_Close', 'Volume_x',
       'Ticker', 'MA_10', 'MA_50', 'RSI_14', 'MACD', 'MACD_Signal', 'BB_Upper',
       'BB_Lower', 'Close_Norm', 'Close_y', 'High_y', 'Low_y', 'Open_y',
       'Volume_y', 'vix', 'High', 'Low', 'Open', 'Volume'],
      dtype='object')
First few rows:
 Empty DataFrame
Columns: [Date, Open_x, High_x, Low_x, Close_x, Adj_Close, Volume_x, Ticker, MA_10, MA_50, RSI_14, MACD, MACD_Signal, BB_Upper, BB_Lower, Close_Norm, Close_y, High_y, Low_y, Open_y, Volume_y, vix, High, Low, Open, Volume]
Index: []

[0 rows x 26 columns]


In [55]:
# List all column names to identify potential issues
print("Column names in DataFrame:", df.columns.tolist())


Column names in DataFrame: ['Date', 'Open_x', 'High_x', 'Low_x', 'Close_x', 'Adj_Close', 'Volume_x', 'Ticker', 'MA_10', 'MA_50', 'RSI_14', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower', 'Close_Norm', 'Close_y', 'High_y', 'Low_y', 'Open_y', 'Volume_y', 'vix', 'High', 'Low', 'Open', 'Volume']


In [57]:
# Rename 'Close' to 'close' for consistency in df
df.rename(columns={"Close": "close"}, inplace=True)

# Check if df is empty and print a warning
if df.empty:
    print("⚠️ Warning: df is empty! Check data loading or merging steps.")

# Print the first few rows of df_vix to verify the column existence
print("First few rows of df_vix:")
print(df_vix.head())

# Check if 'close' column exists in df_vix
if "close" not in df_vix.columns:
    print("'close' column missing from df_vix. Available columns:", df_vix.columns)
else:
    print("'close' column found in df_vix.")



⚠️ Warning: df is empty! Check data loading or merging steps.
First few rows of df_vix:
Price       date        vix       High        Low       Open  Volume
0     2010-01-04  20.040001  21.680000  20.030001  21.680000       0
1     2010-01-05  19.350000  20.129999  19.340000  20.049999       0
2     2010-01-06  19.160000  19.680000  18.770000  19.590000       0
3     2010-01-07  19.059999  19.709999  18.700001  19.680000       0
4     2010-01-08  18.129999  19.270000  18.110001  19.270000       0
'close' column missing from df_vix. Available columns: Index(['date', 'vix', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')


In [59]:
!pip install "shimmy>=2.0" gymnasium

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [60]:
import gym
import numpy as np
import pandas as pd
from gym import spaces
from stable_baselines3 import DDPG

class CustomTradingRiskEnv(gym.Env):
    def __init__(self, df):
        super(CustomTradingRiskEnv, self).__init__()

        self.df = df
        self.current_step = 0
        self.initial_balance = 10000  # Starting cash
        self.cash = self.initial_balance
        self.shares_held = 0
        self.max_steps = len(df) - 1

        # Action space: Continuous action space for DDPG (-1 to 1)
        # Action space: Buy (1), Sell (-1), Hold (0) --> Modified for DDPG
        self.action_space = spaces.Box(low=-1, high=1, shape=(1,), dtype=np.float32)

        # Observation space: [Current Price, Moving Average, RSI, Cash Available, Shares Held, VIX]
        self.observation_space = spaces.Box(
            low=0, high=np.inf, shape=(6,), dtype=np.float32
        )

    def reset(self):
        self.current_step = 0
        self.cash = self.initial_balance
        self.shares_held = 0
        return self._next_observation()

    def _next_observation(self):
        obs = np.array([
            self.df.iloc[self.current_step]["Close"],  # Current price
            self.df.iloc[self.current_step]["SMA_50"],  # 50-day moving average
            self.df.iloc[self.current_step]["RSI"],  # Relative Strength Index
            self.cash,  # Available cash
            self.shares_held,  # Shares held
            self.df.iloc[self.current_step]["vix"]  # VIX (Volatility Index) for risk adjustment
        ])
        return obs

    def step(self, action):
        current_price = self.df.iloc[self.current_step]["Close"]
        vix = self.df.iloc[self.current_step]["vix"]  # Get current VIX for risk analysis

        # Normalize action
        action = action[0]  # Extract the single action value
        reward = 0

        if action > 0:  # Buy action
            shares_to_buy = int(self.cash * action / current_price)
            if shares_to_buy > 0:
                self.shares_held += shares_to_buy
                self.cash -= shares_to_buy * current_price
        elif action < 0:  # Sell action
            shares_to_sell = int(self.shares_held * abs(action))
            if shares_to_sell > 0:
                self.shares_held -= shares_to_sell
                self.cash += shares_to_sell * current_price

        self.current_step += 1

        # Done if we have reached the maximum step
        done = self.current_step >= self.max_steps

        # Calculate risk-adjusted reward (including VIX)
        reward = self.cash + (self.shares_held * current_price) - self.initial_balance

        # Apply VIX penalty: higher VIX means more risk
        vix_penalty = vix / 100  # Normalize VIX
        reward -= vix_penalty  # Penalize for higher volatility

        return self._next_observation(), reward, done, {}



In [61]:
import pandas as pd

# Load Stock Data
df = pd.read_csv("./preprocessed_stock_data.csv")  # Use actual stock data

# Calculate 50-period Simple Moving Average (SMA_50)
df["SMA_50"] = df["Close"].rolling(window=50).mean()

# Calculate 14-period RSI
delta = df["Close"].diff()  # Price difference from previous day
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()  # Average gain over 14 periods
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()  # Average loss over 14 periods

# Avoid division by zero and calculate RS (Relative Strength)
rs = gain / loss
df["RSI"] = 100 - (100 / (1 + rs))

# Display first few rows to verify the calculations
print(df.head())


Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj_Close, Volume, Ticker, MA_10, MA_50, RSI_14, MACD, MACD_Signal, BB_Upper, BB_Lower, Close_Norm, SMA_50, RSI]
Index: []


In [62]:
# Check for NaNs in the DataFrame
print("NaN values in each column:")
print(df.isna().sum())  # This will show the count of NaNs per column

# Remove any rows with NaNs
df = df.dropna()

# Check the first few rows after dropping NaNs
print("\nData after removing NaNs:")
print(df.head())


NaN values in each column:
Date           0.0
Open           0.0
High           0.0
Low            0.0
Close          0.0
Adj_Close      0.0
Volume         0.0
Ticker         0.0
MA_10          0.0
MA_50          0.0
RSI_14         0.0
MACD           0.0
MACD_Signal    0.0
BB_Upper       0.0
BB_Lower       0.0
Close_Norm     0.0
SMA_50         0.0
RSI            0.0
dtype: float64

Data after removing NaNs:
Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj_Close, Volume, Ticker, MA_10, MA_50, RSI_14, MACD, MACD_Signal, BB_Upper, BB_Lower, Close_Norm, SMA_50, RSI]
Index: []


In [63]:
# Assuming you already have df_vix with 'date' and 'vix' columns

# Convert 'date' columns to datetime
df["Date"] = pd.to_datetime(df["Date"])
df_vix["date"] = pd.to_datetime(df_vix["date"])

# Merge VIX data with stock data based on the 'Date' column
df = pd.merge(df, df_vix[["date", "vix"]], left_on="Date", right_on="date", how="left")

# Fill any missing VIX data
df["vix"] = df["vix"].fillna(method="ffill")

# Confirm that VIX is merged properly
print(df.columns.tolist())
print(df.head())


['Date', 'Open', 'High', 'Low', 'Close', 'Adj_Close', 'Volume', 'Ticker', 'MA_10', 'MA_50', 'RSI_14', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower', 'Close_Norm', 'SMA_50', 'RSI', 'date', 'vix']
Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj_Close, Volume, Ticker, MA_10, MA_50, RSI_14, MACD, MACD_Signal, BB_Upper, BB_Lower, Close_Norm, SMA_50, RSI, date, vix]
Index: []


In [69]:
class CustomTradingRiskEnv(gym.Env):
    def __init__(self, df):
        super(CustomTradingRiskEnv, self).__init__()

        self.df = df
        self.current_step = 0
        self.initial_balance = 10000  # Starting cash
        self.cash = self.initial_balance
        self.shares_held = 0
        self.max_steps = len(df) - 1

        # Action space: Continuous action space for DDPG (-1 to 1)
        self.action_space = spaces.Box(low=-1, high=1, shape=(1,), dtype=np.float32)

        # Observation space: [Current Price, Moving Average, RSI, Cash Available, Shares Held, VIX]
        self.observation_space = spaces.Box(
            low=0, high=np.inf, shape=(6,), dtype=np.float32
        )

    def reset(self):
        self.current_step = 0
        self.cash = self.initial_balance
        self.shares_held = 0
        return self._next_observation()

    def _next_observation(self):
        # Check if 'vix' exists in the dataframe
        if 'vix' not in self.df.columns:
            print("Error: 'vix' column is missing!")
            return None

        obs = np.array([
            self.df.iloc[self.current_step]["Close"],  # Current price
            self.df.iloc[self.current_step]["SMA_50"],  # 50-day moving average
            self.df.iloc[self.current_step]["RSI"],  # Relative Strength Index
            self.cash,  # Available cash
            self.shares_held,  # Shares held
            self.df.iloc[self.current_step]["vix"]  # VIX (Volatility Index) for risk adjustment
        ])
        return obs

    def step(self, action):
        current_price = self.df.iloc[self.current_step]["Close"]
        vix = self.df.iloc[self.current_step]["vix"]  # Get current VIX for risk analysis

        # Normalize action
        action = action[0]  # Extract the single action value
        reward = 0

        if action > 0:  # Buy action
            shares_to_buy = int(self.cash * action / current_price)
            if shares_to_buy > 0:
                self.shares_held += shares_to_buy
                self.cash -= shares_to_buy * current_price
        elif action < 0:  # Sell action
            shares_to_sell = int(self.shares_held * abs(action))
            if shares_to_sell > 0:
                self.shares_held -= shares_to_sell
                self.cash += shares_to_sell * current_price

        self.current_step += 1

        # Done if we have reached the maximum step
        done = self.current_step >= self.max_steps

        # Calculate risk-adjusted reward (including VIX)
        reward = self.cash + (self.shares_held * current_price) - self.initial_balance

        # Apply VIX penalty: higher VIX means more risk
        vix_penalty = vix / 100  # Normalize VIX
        reward -= vix_penalty  # Penalize for higher volatility

        return self._next_observation(), reward, done, {}


In [77]:
print("Checking raw data before preprocessing:")
print(df.head())  # Display first few rows
print("DataFrame shape:", df.shape)


Checking raw data before preprocessing:
Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj_Close, Volume, Ticker, MA_10, MA_50, RSI_14, MACD, MACD_Signal, BB_Upper, BB_Lower, Close_Norm, SMA_50, RSI, date, vix]
Index: []
DataFrame shape: (0, 20)


In [73]:
# Train the DDPG Model
env = CustomTradingRiskEnv(df)
model = DDPG("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10000)

#  Evaluate the Model
obs = env.reset()
done = False
total_reward = 0  # To track the total reward during evaluation
step_count = 0

while not done:
    action, _ = model.predict(obs)  # Get the action from the model
    action = np.clip(action, -1, 1)  # Ensure action is within valid range for DDPG (-1 to 1)
    obs, reward, done, _ = env.step(action)  # Step through the environment with the action
    
    total_reward += reward  # Accumulate the reward
    step_count += 1  # Track the number of steps taken
    
    if step_count % 100 == 0:
        print(f"Step: {step_count}, Total Reward: {total_reward}")

# Calculate and print the final portfolio value
final_value = env.cash + (env.shares_held * df.iloc[-1]["Close"])
print(f"Final Portfolio Value: {final_value}")


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


C:\Users\aswin\anaconda3\envs\finrl_env\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


IndexError: single positional indexer is out-of-bounds

## Improve Risk Awareness in DDGP

In [326]:
def _calculate_reward(self):
    """
    Calculate reward based on portfolio value, risk-awareness, and performance metrics.
    - Penalizes high volatility and large fluctuations in returns.
    """
    
    # Current portfolio value
    portfolio_value = self.cash + self.shares_held * self.df.iloc[self.current_step]["Close"]
    
    # Portfolio return relative to the initial investment
    portfolio_return = portfolio_value / self.initial_cash
    
    # Calculate daily return for risk management
    if self.current_step > 0:
        daily_return = (portfolio_value - self.previous_portfolio_value) / self.previous_portfolio_value
    else:
        daily_return = 0  # No return at the start
    
    self.previous_portfolio_value = portfolio_value
    
    # Calculate risk (Penalize large fluctuations)
    risk_penalty = - abs(daily_return)  # Larger the daily return, larger the penalty
    
    # Calculate Sharpe Ratio (Risk-adjusted return)
    if len(self.returns) > 1:
        sharpe_ratio = np.mean(self.returns) / (np.std(self.returns) + 1e-6)  # Avoid div by zero
    else:
        sharpe_ratio = 0  # Default value when not enough data for Sharpe ratio
    
    # Reward calculation: portfolio return + Sharpe ratio + risk penalty
    reward = portfolio_return + 0.1 * sharpe_ratio + risk_penalty

    # Optionally, scale or clip reward to avoid large outliers
    reward = np.clip(reward, -1, 1)  # Clip the reward to be between -1 and 1 (for example)
    
    return reward


## Adjust DDGP Hyperparameters

In [332]:
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv

# Action noise for continuous actions
action_noise = NormalActionNoise(mean=np.zeros(env.action_space.shape[0]), sigma=0.1 * np.ones(env.action_space.shape[0]))

# Initialize the DDPG model with adjusted hyperparameters
policy_kwargs = dict(
    net_arch=[400, 300]  # Define the neural network architecture here
)

model = DDPG(
    "MlpPolicy",  # Policy network type
    env,  # Environment
    action_noise=action_noise,  # Action noise for exploration
    verbose=1,  # Set to 1 for more output
    learning_rate=0.0001,  # Learning rate for the optimizer
    gamma=0.99,  # Discount factor
    tau=0.005,  # Soft target updates (similar to Polyak averaging)
    batch_size=64,  # Batch size for training
    buffer_size=1000000,  # Size of the replay buffer
    learning_starts=100,  # How many time steps before starting learning
    train_freq=1,  # Frequency of model updates
    gradient_steps=1,  # How many gradient steps to perform after each rollout
    policy_kwargs=policy_kwargs  # Pass the network architecture here
)

# Train the DDPG Model
model.learn(total_timesteps=10000)

# Evaluate the Model
obs = env.reset()
done = False
while not done:
    action, _ = model.predict(obs)
    obs, reward, done, _ = env.step(action)

print("Final Portfolio Value:", env.cash + (env.shares_held * df.iloc[-1]["Close"]))


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


C:\Users\aswin\anaconda3\envs\finrl_env\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Final Portfolio Value: 10000.0


In [335]:
# Increase action noise to encourage more exploration
action_noise = NormalActionNoise(mean=np.zeros(env.action_space.shape[0]), sigma=0.2 * np.ones(env.action_space.shape[0]))

# Initialize the DDPG model with adjusted hyperparameters
model = DDPG(
    "MlpPolicy",  # Policy network type
    env,  # Environment
    action_noise=action_noise,  # Action noise for exploration
    verbose=1,  # Set to 1 for more output
    learning_rate=0.0001,  # Learning rate for the optimizer
    gamma=0.99,  # Discount factor
    tau=0.005,  # Soft target updates (similar to Polyak averaging)
    batch_size=64,  # Batch size for training
    buffer_size=1000000,  # Size of the replay buffer
    learning_starts=100,  # How many time steps before starting learning
    train_freq=1,  # Frequency of model updates
    gradient_steps=1,  # How many gradient steps to perform after each rollout
    policy_kwargs=policy_kwargs  # Pass the network architecture here
)

# Train the DDPG Model with more timesteps
model.learn(total_timesteps=100000)

# Evaluate the Model
obs = env.reset()
done = False
while not done:
    action, _ = model.predict(obs)
    obs, reward, done, _ = env.step(action)

print("Final Portfolio Value:", env.cash + (env.shares_held * df.iloc[-1]["Close"]))


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


C:\Users\aswin\anaconda3\envs\finrl_env\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Final Portfolio Value: 142035.05350351337


In [339]:
%matplotlib inline


In [341]:
import matplotlib
matplotlib.use('TkAgg')  # This sets the interactive backend

import matplotlib.pyplot as plt


In [345]:
plt.plot(portfolio_values)
plt.title('Portfolio Value Over Time')
plt.xlabel('Time Steps')
plt.ylabel('Portfolio Value')
plt.savefig('portfolio_value.png')  # Save the plot to a file
plt.close()  # Close the plot to avoid memory usage buildup


## EVALUATE THE MODEL

### COMPUTE THE KEY PERFORMANCE METRICS

In [3]:
import numpy as np

# Calculate total portfolio return
final_portfolio_value = env.cash + (env.shares_held * df.iloc[-1]["Close"])
total_return = (final_portfolio_value - env.initial_balance) / env.initial_balance

# Compute Sharpe Ratio (Risk-Adjusted Return)
returns = np.array(env.returns)
sharpe_ratio = np.mean(returns) / (np.std(returns) + 1e-6)  # Avoid division by zero

# Compute Maximum Drawdown
cumulative_returns = np.cumsum(returns)
running_max = np.maximum.accumulate(cumulative_returns)
drawdowns = running_max - cumulative_returns
max_drawdown = np.max(drawdowns)

print(f"Final Portfolio Value: {final_portfolio_value}")
print(f"Total Return: {total_return * 100:.2f}%")
print(f"Sharpe Ratio: {sharpe_ratio:.3f}")
print(f"Maximum Drawdown: {max_drawdown:.2f}")


NameError: name 'env' is not defined